# Dados Meteorológicos

Este notebook tem duas finalidades:
1. Apresentar os metadados dos dados meteorológicos disponíveis no data lake do projeto.
1. Apresentar exemplos de acesso aos dados empregando python.

# Visão Geral

O data lake está dividido em três buckets:

1. `landing`
    * Dados brutos, sem qualquer processamento.
2. `staged`
    * Dados otimizados.
3. `curated`
    * Dados curados, prontos para serem consumidos.

Este notebook apresenta os dados meteorológicos curados, _i.e._, os dados disponíveis no bucket `curated`. 

A exceção é a seção `Informações dos Instrumentos Meteorológicos`, que carrega os dados do bucket `landing`.

# Data Lake Wrapper

In [1]:
import os
import pandas as pd
from pyarrow import fs, parquet as pq
from getpass import getpass


class DataLakeWrapper():
    
    def __init__(self) -> None:
        self._login()
        self._filesystem = self._build_filesystem()

    
    def read_parquet_dataset(self, path, filters=None):
        table = pq.ParquetDataset(
            path,
            filters=filters,
            filesystem=self._filesystem,
        ).read()
        return table
    
    
    def read_parquet_table(self, path, filters=None, columns=None):
        table = pq.read_table(
            path,
            filters=filters,
            columns=columns,
            filesystem=self._filesystem
        )
        return table

    
    def _login(self):
        os.environ['MINIO_ENDPOINT'] = getpass('Enter the Minio endpoint: ')
        os.environ['MINIO_USER'] = getpass('Enter the Minio user: ')
        os.environ['MINIO_PASSWORD'] = getpass('Enter the Minio password: ')
    

    def _build_filesystem(self):
        return fs.S3FileSystem(
            endpoint_override=os.getenv('MINIO_ENDPOINT'),
            access_key=os.getenv('MINIO_USER'),
            secret_key=os.getenv('MINIO_PASSWORD'),
            scheme='http'
        )

In [2]:
dl = DataLakeWrapper()

Enter the Minio endpoint: ········
Enter the Minio user: ········
Enter the Minio password: ········


# Informações dos Instrumentos Meteorológicos

## Estações do Alerta Rio

In [3]:
path = 'landing/instruments_info/alertario_stations.parquet'
df = dl.read_parquet_table(path).to_pandas()
df

,id_estacao,estacao,estacao_desc,latitude,longitude,cota,x,y
0,8,Ilha do governador,ilha_do_governador,-22.81806,-43.21028,0,6.837087e+08,7.475960e+09
1,20,Guaratiba,guaratiba,-23.05028,-43.59472,0,6.439722e+08,7.450214e+06
2,16,Jardim botanico,jardim_botanico,-22.97278,-43.22389,0,6.821335e+05,7.458453e+09
3,19,Riocentro,riocentro,-22.97721,-43.39155,0,6.648794e+05,7.458100e+06
4,17,Barrinha,barrinha,-23.00849,-43.29965,7,6.742621e+08,7.454521e+09
5,30,Recreio,recreio,-23.01000,-43.44056,10,6.598168e+08,7.454514e+09
6,25,Grota funda,grota_funda,-23.01444,-43.52139,11,6.515264e+08,7.454108e+09
7,15,Saude,saude,-22.89606,-43.18786,15,6.858751e+08,7.466833e+09
8,22,Santa cruz,santa_cruz,-22.90944,-43.68444,15,6.349159e+08,7.465594e+09
9,18,Cidade de deus,cidade_de_deus,-22.94556,-43.36278,15,6.679282e+08,7.461633e+09


## Estações do INMET

In [4]:
path = 'landing/instruments_info/inmet_stations.parquet'
df = dl.read_parquet_table(path).to_pandas()
df

,cd_estacao,dc_nome,dc_nome_desc,altitude,latitude,longitude,dt_inicio_operacao
0,A602,Marambaia,marambaia,12.00,-23.050278,-43.595556,2002-11-07 22:00:00-02:00
1,A621,Vila Militar,vila_militar,30.43,-22.861389,-43.411389,2007-04-12 21:00:00-03:00
2,A652,Forte De Copacabana,forte_de_copacabana,25.59,-22.988333,-43.190556,2007-05-17 21:00:00-03:00
3,A636,Jacarepagua,jacarepagua,20.00,-22.940000,-43.402778,2017-08-09 21:00:00-03:00


## Estações do sistema Websirenes

In [5]:
path = 'landing/instruments_info/websirenes_stations.parquet'
with pd.option_context('display.max_rows', None,):
    df = dl.read_parquet_table(path).to_pandas()
    display(df)

,id_estacao,estacao,estacao_desc,latitude,longitude
0,33,Ladeira dos Tabajaras,ladeira_dos_tabajaras,-22.961700,-43.188000
1,8,Cabritos 1,cabritos_1,-22.964700,-43.195000
2,26,Guararapes 1,guararapes_1,-22.944700,-43.208000
3,34,Liberdade 1,liberdade_1,-22.926600,-43.218000
4,64,Salgueiro 1,salgueiro_1,-22.930200,-43.226000
5,38,Matriz 1,matriz_1,-22.904600,-43.264000
6,47,Palmeiras 2,palmeiras_2,-22.864100,-43.283000
7,32,Juramento 2,juramento_2,-22.857500,-43.313000
8,72,Sapê 1,sape_1,-22.857500,-43.333000
9,17,Chapéu Mangueira 1,chapeu_mangueira_1,-22.960500,-43.167900


# Metadados

## Pluviômetros do Alerta Rio

In [6]:
path = 'curated/rain_gauge/alertario'
table = dl.read_parquet_dataset(path)
table.schema

station: string
datetime: timestamp[us, tz=UTC]
precipitation: double
hour_sin: double
hour_cos: double
month_sin: double
month_cos: double
latitude: double
longitude: double
year: dictionary<values=int32, indices=int32, ordered=0>
month: dictionary<values=int32, indices=int32, ordered=0>
-- schema metadata --
naming_authority: 'Alerta Rio'
timezone: 'UTC'
instrument: 'rain gauge'
variable_1: 'station - station name'
variable_2: 'datetime - UTC datetime of the measurement'
variable_3: 'precipitation (mm/15min)'
variable_4: 'hour_sin - sine encoding of the time of day'
variable_5: 'hour_cos - cosine encoding of the time of day'
variable_6: 'month_sin - sine encoding of the month of year'
variable_7: 'month_cos - cosine encoding of the month of year'
variable_8: 'latitude (degrees)'
variable_9: 'longitude (degrees)'

In [7]:
df = table.to_pandas()
df

,station,datetime,precipitation,hour_sin,hour_cos,month_sin,month_cos,latitude,longitude,year,month
0,vidigal,2016-01-01 02:00:00+00:00,0.0,0.500000,0.866025,0.5,8.660254e-01,-22.99250,-43.23306,2016,1
1,vidigal,2016-01-01 02:15:00+00:00,0.0,0.555570,0.831470,0.5,8.660254e-01,-22.99250,-43.23306,2016,1
2,vidigal,2016-01-01 02:30:00+00:00,0.0,0.608761,0.793353,0.5,8.660254e-01,-22.99250,-43.23306,2016,1
3,vidigal,2016-01-01 02:45:00+00:00,0.0,0.659346,0.751840,0.5,8.660254e-01,-22.99250,-43.23306,2016,1
4,vidigal,2016-01-01 03:00:00+00:00,0.0,0.707107,0.707107,0.5,8.660254e-01,-22.99250,-43.23306,2016,1
...,...,...,...,...,...,...,...,...,...,...,...
8268685,tijuca_muda,2023-03-10 04:00:00+00:00,0.0,0.866025,0.500000,1.0,6.123234e-17,-22.93278,-43.24333,2023,3
8268686,tijuca_muda,2023-03-10 04:15:00+00:00,0.0,0.896873,0.442289,1.0,6.123234e-17,-22.93278,-43.24333,2023,3
8268687,tijuca_muda,2023-03-10 04:30:00+00:00,0.0,0.923880,0.382683,1.0,6.123234e-17,-22.93278,-43.24333,2023,3
8268688,tijuca_muda,2023-03-10 04:45:00+00:00,0.0,0.946930,0.321439,1.0,6.123234e-17,-22.93278,-43.24333,2023,3


In [8]:
df['datetime'].describe()

count                                8268690
mean     2019-07-30 05:41:04.829748992+00:00
min                2016-01-01 02:00:00+00:00
25%                2017-10-14 14:15:00+00:00
50%                2019-07-29 04:00:00+00:00
75%                2021-05-11 16:30:00+00:00
max                2023-03-10 05:00:00+00:00
Name: datetime, dtype: object

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8268690 entries, 0 to 8268689
Data columns (total 11 columns):
 #   Column         Dtype              
---  ------         -----              
 0   station        object             
 1   datetime       datetime64[ns, UTC]
 2   precipitation  float64            
 3   hour_sin       float64            
 4   hour_cos       float64            
 5   month_sin      float64            
 6   month_cos      float64            
 7   latitude       float64            
 8   longitude      float64            
 9   year           category           
 10  month          category           
dtypes: category(2), datetime64[ns, UTC](1), float64(7), object(1)
memory usage: 583.5+ MB


## Estações Meteorológicas do Alerta Rio

In [10]:
path = 'curated/weather_station/alertario'
table = dl.read_parquet_dataset(path)
table.schema

station: string
datetime: timestamp[us, tz=UTC]
precipitation: double
wind_dir: double
wind_speed: double
temperature: double
pressure: double
humidity: double
wind_u: double
wind_v: double
hour_sin: double
hour_cos: double
month_sin: double
month_cos: double
latitude: double
longitude: double
year: dictionary<values=int32, indices=int32, ordered=0>
month: dictionary<values=int32, indices=int32, ordered=0>
-- schema metadata --
naming_authority: 'Alerta Rio'
timezone: 'UTC'
instrument: 'weather station'
variable_1: 'station - station name'
variable_2: 'datetime - UTC datetime of the measurement'
variable_3: 'precipitation (mm/15min)'
variable_4: 'wind_dir (degrees)'
variable_5: 'wind_speed (km/h)'
variable_6: 'temperature (degrees Celsius)'
variable_7: 'pressure (hPa)'
variable_8: 'humidity (%)'
variable_9: 'wind_u (cyclic U component from the wind)'
variable_10: 'wind_v (cyclic V component from the wind)'
variable_11: 'hour_sin (sine encoding of the time of day)'
variable_12: 'hour_co

In [11]:
df = table.to_pandas()
df

,station,datetime,precipitation,wind_dir,wind_speed,temperature,pressure,humidity,wind_u,wind_v,hour_sin,hour_cos,month_sin,month_cos,latitude,longitude,year,month
0,iraja,2016-01-01 02:00:00+00:00,0.0,NaN,NaN,30.8,NaN,60.0,NaN,NaN,0.500000,0.866025,0.500000,0.866025,-22.82694,-43.33694,2016,1
1,iraja,2016-01-01 02:15:00+00:00,0.0,NaN,NaN,30.8,NaN,59.0,NaN,NaN,0.555570,0.831470,0.500000,0.866025,-22.82694,-43.33694,2016,1
2,iraja,2016-01-01 02:30:00+00:00,0.0,NaN,NaN,30.6,NaN,60.0,NaN,NaN,0.608761,0.793353,0.500000,0.866025,-22.82694,-43.33694,2016,1
3,iraja,2016-01-01 02:45:00+00:00,0.0,NaN,NaN,30.8,NaN,60.0,NaN,NaN,0.659346,0.751840,0.500000,0.866025,-22.82694,-43.33694,2016,1
4,iraja,2016-01-01 03:00:00+00:00,0.0,NaN,NaN,30.6,NaN,61.0,NaN,NaN,0.707107,0.707107,0.500000,0.866025,-22.82694,-43.33694,2016,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1913632,vidigal,2023-04-01 01:45:00+00:00,0.0,340.0,5.8,NaN,NaN,NaN,1.983717,-5.450217,0.442289,0.896873,0.866025,-0.500000,-22.99250,-43.23306,2023,4
1913633,vidigal,2023-04-01 02:00:00+00:00,0.0,325.0,8.6,NaN,NaN,NaN,4.932757,-7.044708,0.500000,0.866025,0.866025,-0.500000,-22.99250,-43.23306,2023,4
1913634,vidigal,2023-04-01 02:15:00+00:00,0.0,331.0,11.5,NaN,NaN,NaN,5.575311,-10.058127,0.555570,0.831470,0.866025,-0.500000,-22.99250,-43.23306,2023,4
1913635,vidigal,2023-04-01 02:30:00+00:00,0.0,334.0,11.5,NaN,NaN,NaN,5.041268,-10.336132,0.608761,0.793353,0.866025,-0.500000,-22.99250,-43.23306,2023,4


In [12]:
df['datetime'].describe()

count                                1913637
mean     2019-10-08 08:37:58.472877056+00:00
min                2016-01-01 02:00:00+00:00
25%                2017-12-22 22:45:00+00:00
50%                2019-12-23 07:15:00+00:00
75%                2021-07-10 01:15:00+00:00
max                2023-04-01 02:45:00+00:00
Name: datetime, dtype: object

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1913637 entries, 0 to 1913636
Data columns (total 18 columns):
 #   Column         Dtype              
---  ------         -----              
 0   station        object             
 1   datetime       datetime64[ns, UTC]
 2   precipitation  float64            
 3   wind_dir       float64            
 4   wind_speed     float64            
 5   temperature    float64            
 6   pressure       float64            
 7   humidity       float64            
 8   wind_u         float64            
 9   wind_v         float64            
 10  hour_sin       float64            
 11  hour_cos       float64            
 12  month_sin      float64            
 13  month_cos      float64            
 14  latitude       float64            
 15  longitude      float64            
 16  year           category           
 17  month          category           
dtypes: category(2), datetime64[ns, UTC](1), float64(14), object(1)
memory usage: 237.2

## Estações Meteorológicas do INMET

In [14]:
path = 'curated/weather_station/inmet'
table = dl.read_parquet_dataset(path)
table.schema

station_id: string
datetime: timestamp[us, tz=UTC]
precipitation: double
pressure: double
temperature: double
dew_point: double
humidity: double
wind_dir: double
wind_speed: double
wind_u: double
wind_v: double
hour_sin: double
hour_cos: double
month_sin: double
month_cos: double
station: string
latitude: double
longitude: double
year: dictionary<values=int32, indices=int32, ordered=0>
month: dictionary<values=int32, indices=int32, ordered=0>
-- schema metadata --
naming_authority: 'INMET - Instituto Nacional de Meteorologia'
timezone: 'UTC'
instrument: 'weather station'
variable_1: 'station_id (station UID)'
variable_2: 'datetime (datetime of the measurement)'
variable_3: 'precipitation (hourly precipitation in mm)'
variable_4: 'pressure (instant atmospheric pressure in mB)'
variable_5: 'temperature (instant temperature in Celsius)'
variable_6: 'dew_point (dew point temperature in Celsius)'
variable_7: 'humidity (instant relative humidity in %)'
variable_8: 'wind_dir (clockwise wind d

In [15]:
df = table.to_pandas()
df

,station_id,datetime,precipitation,pressure,temperature,dew_point,humidity,wind_dir,wind_speed,wind_u,wind_v,hour_sin,hour_cos,month_sin,month_cos,station,latitude,longitude,year,month
0,A602,2002-11-08 00:00:00+00:00,0.0,1021.4,18.2,15.7,86.0,28.0,2.0,-0.938943,-1.765895,0.000000,1.000000,-0.5,8.660254e-01,marambaia,-23.050278,-43.595556,2002,11
1,A602,2002-11-08 01:00:00+00:00,0.0,1021.9,18.5,16.2,87.0,348.0,2.5,0.519779,-2.445369,0.258819,0.965926,-0.5,8.660254e-01,marambaia,-23.050278,-43.595556,2002,11
2,A602,2002-11-08 02:00:00+00:00,3.6,1021.7,17.8,15.9,89.0,17.0,2.5,-0.730929,-2.390762,0.500000,0.866025,-0.5,8.660254e-01,marambaia,-23.050278,-43.595556,2002,11
3,A602,2002-11-08 03:00:00+00:00,0.0,1020.9,17.4,15.7,90.0,29.0,1.8,-0.872657,-1.574315,0.707107,0.707107,-0.5,8.660254e-01,marambaia,-23.050278,-43.595556,2002,11
4,A602,2002-11-08 04:00:00+00:00,0.0,1020.3,17.2,15.6,91.0,2.0,2.4,-0.083759,-2.398538,0.866025,0.500000,-0.5,8.660254e-01,marambaia,-23.050278,-43.595556,2002,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533707,A621,2023-09-30 19:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.965926,0.258819,-1.0,-1.836970e-16,vila_militar,-22.861389,-43.411389,2023,9
533708,A621,2023-09-30 20:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.866025,0.500000,-1.0,-1.836970e-16,vila_militar,-22.861389,-43.411389,2023,9
533709,A621,2023-09-30 21:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.707107,0.707107,-1.0,-1.836970e-16,vila_militar,-22.861389,-43.411389,2023,9
533710,A621,2023-09-30 22:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.500000,0.866025,-1.0,-1.836970e-16,vila_militar,-22.861389,-43.411389,2023,9


In [16]:
df['datetime'].describe()

count                                 533712
mean     2015-06-05 11:15:45.246874368+00:00
min                2002-11-08 00:00:00+00:00
25%                2010-11-28 19:45:00+00:00
50%                2015-12-25 23:30:00+00:00
75%                2020-03-12 02:15:00+00:00
max                2023-12-31 23:00:00+00:00
Name: datetime, dtype: object

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 533712 entries, 0 to 533711
Data columns (total 20 columns):
 #   Column         Non-Null Count   Dtype              
---  ------         --------------   -----              
 0   station_id     533712 non-null  object             
 1   datetime       533712 non-null  datetime64[ns, UTC]
 2   precipitation  483275 non-null  float64            
 3   pressure       482080 non-null  float64            
 4   temperature    483451 non-null  float64            
 5   dew_point      453380 non-null  float64            
 6   humidity       463563 non-null  float64            
 7   wind_dir       461738 non-null  float64            
 8   wind_speed     470152 non-null  float64            
 9   wind_u         461738 non-null  float64            
 10  wind_v         461738 non-null  float64            
 11  hour_sin       533712 non-null  float64            
 12  hour_cos       533712 non-null  float64            
 13  month_sin      533712 non-nul

## Radar Meteorológico do INEA

In [18]:
path = 'curated/radar/inea/guaratiba'
table = dl.read_parquet_dataset(path)
table.schema

datetime: timestamp[us, tz=UTC]
latitude: double
longitude: double
altitude: int64
horizontal_reflectivity_mean: double
hour_sin: double
hour_cos: double
month_sin: double
month_cos: double
year: dictionary<values=int32, indices=int32, ordered=0>
month: dictionary<values=int32, indices=int32, ordered=0>
day: dictionary<values=int32, indices=int32, ordered=0>
-- schema metadata --
naming_authority: 'INEA - Instituto Estadual do Ambiente'
timezone: 'UTC'
instrument: 'radar'
variable_1: 'longitude (degrees)'
variable_2: 'latitude (degrees)'
variable_3: 'altitude (kilometers)'
variable_4: 'horizontal reflectivity (dBZ)'
variable_5: 'hour_sin - sine encoding of the time of day'
variable_6: 'hour_cos - cosine encoding of the time of day'
variable_7: 'month_sin - sine encoding of the month of year'
variable_8: 'month_cos - cosine encoding of the month of year'

In [19]:
df = table.to_pandas()
df

,datetime,latitude,longitude,altitude,horizontal_reflectivity_mean,hour_sin,hour_cos,month_sin,month_cos,year,month,day
0,2016-11-12 00:00:00+00:00,-23.02,-43.65,0,11.836508,0.000000,1.000000,-0.5,0.866025,2016,11,12
1,2016-11-12 00:00:00+00:00,-23.02,-43.64,0,8.429251,0.000000,1.000000,-0.5,0.866025,2016,11,12
2,2016-11-12 00:00:00+00:00,-23.01,-43.63,0,10.506431,0.000000,1.000000,-0.5,0.866025,2016,11,12
3,2016-11-12 00:00:00+00:00,-23.00,-43.59,0,27.785027,0.000000,1.000000,-0.5,0.866025,2016,11,12
4,2016-11-12 00:00:00+00:00,-23.01,-43.62,0,6.814247,0.000000,1.000000,-0.5,0.866025,2016,11,12
...,...,...,...,...,...,...,...,...,...,...,...,...
120303112,2023-01-03 23:00:00+00:00,-22.65,-43.60,4,8.291667,-0.258819,0.965926,0.5,0.866025,2023,1,3
120303113,2023-01-03 23:00:00+00:00,-22.68,-43.59,4,10.000000,-0.258819,0.965926,0.5,0.866025,2023,1,3
120303114,2023-01-03 23:00:00+00:00,-22.67,-43.59,4,6.645833,-0.258819,0.965926,0.5,0.866025,2023,1,3
120303115,2023-01-03 23:00:00+00:00,-22.66,-43.59,4,7.100000,-0.258819,0.965926,0.5,0.866025,2023,1,3


In [20]:
df['datetime'].describe()

count                              120303117
mean     2019-02-11 06:00:35.513782272+00:00
min                2016-09-19 00:00:00+00:00
25%                2017-01-24 21:00:00+00:00
50%                2018-11-25 03:00:00+00:00
75%                2021-02-05 21:00:00+00:00
max                2023-01-19 23:00:00+00:00
Name: datetime, dtype: object

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120303117 entries, 0 to 120303116
Data columns (total 12 columns):
 #   Column                        Dtype              
---  ------                        -----              
 0   datetime                      datetime64[ns, UTC]
 1   latitude                      float64            
 2   longitude                     float64            
 3   altitude                      int64              
 4   horizontal_reflectivity_mean  float64            
 5   hour_sin                      float64            
 6   hour_cos                      float64            
 7   month_sin                     float64            
 8   month_cos                     float64            
 9   year                          category           
 10  month                         category           
 11  day                           category           
dtypes: category(3), datetime64[ns, UTC](1), float64(7), int64(1)
memory usage: 8.4 GB


# Exemplo de Carga com Filtros
https://arrow.apache.org/docs/python/generated/pyarrow.parquet.ParquetDataset.html

In [22]:
filters = [
    ['year', '=', 2017],
    ['month', 'in', [2, 3]]
]
path = 'curated/radar/inea/guaratiba'
df = dl.read_parquet_dataset(path, filters=filters).to_pandas()
df

,datetime,latitude,longitude,altitude,horizontal_reflectivity_mean,hour_sin,hour_cos,month_sin,month_cos,year,month,day
0,2017-03-01 00:00:00+00:00,-22.99,-43.59,0,48.162200,0.000000,1.000000,1.0,6.123234e-17,2017,3,1
1,2017-03-01 00:00:00+00:00,-22.98,-43.59,0,14.915751,0.000000,1.000000,1.0,6.123234e-17,2017,3,1
2,2017-03-01 00:00:00+00:00,-22.97,-43.59,0,13.297158,0.000000,1.000000,1.0,6.123234e-17,2017,3,1
3,2017-03-01 00:00:00+00:00,-22.96,-43.59,0,10.174603,0.000000,1.000000,1.0,6.123234e-17,2017,3,1
4,2017-03-01 00:00:00+00:00,-22.95,-43.59,0,16.419048,0.000000,1.000000,1.0,6.123234e-17,2017,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...
9935471,2017-03-08 23:00:00+00:00,-22.97,-43.32,9,0.000000,-0.258819,0.965926,1.0,6.123234e-17,2017,3,8
9935472,2017-03-08 23:00:00+00:00,-22.96,-43.21,14,0.000000,-0.258819,0.965926,1.0,6.123234e-17,2017,3,8
9935473,2017-03-08 23:00:00+00:00,-22.96,-43.21,13,0.000000,-0.258819,0.965926,1.0,6.123234e-17,2017,3,8
9935474,2017-03-08 23:00:00+00:00,-22.96,-43.22,13,0.000000,-0.258819,0.965926,1.0,6.123234e-17,2017,3,8


In [23]:
display(df['year'].unique(), df['month'].unique())

[2017]
Categories (8, int32): [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

[3]
Categories (12, int32): [11, 12, 9, 1, ..., 7, 8, 2, 5]

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9935476 entries, 0 to 9935475
Data columns (total 12 columns):
 #   Column                        Dtype              
---  ------                        -----              
 0   datetime                      datetime64[ns, UTC]
 1   latitude                      float64            
 2   longitude                     float64            
 3   altitude                      int64              
 4   horizontal_reflectivity_mean  float64            
 5   hour_sin                      float64            
 6   hour_cos                      float64            
 7   month_sin                     float64            
 8   month_cos                     float64            
 9   year                          category           
 10  month                         category           
 11  day                           category           
dtypes: category(3), datetime64[ns, UTC](1), float64(7), int64(1)
memory usage: 710.6 MB
